# Qwen3.5 recurrent state: 매 토큰 FP32 수집 + 3D Explorer

**모델:** `Qwen/Qwen3.5-0.8B` · **기본값:** 실제 문서 256 tokens, layer 0/8/16 × head 0/1 · **매 토큰:** 128×128 FP32 행렬 전체 저장.

기존 `qwen_recurrent_state_gpu.ipynb`를 읽어 수정한 완성본입니다. 원본의 모델 revision, 예시 문서, 실험 1의 분포·SVD 분석, 실험 3의 축별 static quantization을 유지합니다. **실험 2는 64토큰 간격의 요약 통계에서 매 토큰 실제 state를 보는 3D surface로 확장**했습니다. 기존 시간별 통계도 남아 있습니다.

## 빠른 사용법

1. Colab에 **이 `.ipynb` 하나**를 업로드 → `런타임 → 런타임 유형 변경 → GPU`.
2. 설치 셀 실행. 이미 Transformers/Plotly를 import한 세션이면 설치 후 런타임을 재시작하고 다음 셀부터 실행합니다.
3. 설정의 `TEXT`, `MAX_TOKENS`, `LAYERS`, `HEADS`를 확인한 후 위에서부터 실행합니다.
4. 수집 → 정확성 검증 → NPZ 저장 후 3D Explorer에서 시간·layer·head를 선택합니다. ▶로 재생하고 드래그/휠로 회전·줌합니다.
5. 마지막 셀의 버튼으로 NPZ 또는 결과 ZIP을 다운로드합니다. 이후 NPZ만 다시 읽어 GPU 없이 시각화할 수 있습니다.

**검증 범위:** 작성 환경에서 Transformers 5.24.0의 작은 무작위 초기화 CPU 모델로 실제 cache 연동·매 토큰 수집·prefix 비교·NPZ 왕복을 실행 검증했습니다. 위젯 조작과 3D HTML 렌더링도 확인했습니다. **사전학습된 0.8B 모델의 Colab GPU 전체 실행은 작성 환경에서 수행하지 않았습니다.** 실제 실험값과 GPU 시간/메모리는 아래 셀에서 실행 시 측정합니다. 미리 채워 넣은 모델 결과는 없습니다.

## 1. 설치

Colab의 기존 CUDA PyTorch를 사용합니다. `flash-linear-attention` / `causal-conv1d`는 설치하지 않습니다. 이 측정에서는 검증한 native PyTorch 연산을 사용합니다. 외부 커널이 이미 설치되어 있으면 로더가 이를 감지하고 중단하므로 새 런타임에서 시작하십시오.

In [ ]:
%pip -q install "transformers==5.24.0" "plotly==6.5.2" "ipywidgets==8.1.9" "anywidget==0.11.0" accelerate pillow matplotlib pandas


## 2. 공통 함수 준비

다음 세 셀은 모델 로딩/기존 분석, 실제 state 수집/검증/NPZ, 3D 뷰어 코드를 현재 폴더에 씁니다. 별도 Python 파일을 업로드할 필요가 없습니다. 수정하는 사용자 설정은 바로 아래 **3절**에 모았습니다.

In [ ]:
%%writefile recurrent_gpu_probe.py
"""Small-GPU diagnostics for a pretrained Qwen3.5-0.8B recurrent state.

This is a measurement script, not a trained quantization method. The author's
environment did not run the pretrained GPU path; native cache integration is
validated separately on a tiny randomly initialized CPU model. Run the accompanying notebook on a Colab GPU to obtain real data.
"""
from __future__ import annotations

import csv
import hashlib
import json
import math
import platform
import time
from pathlib import Path

import numpy as np

MODEL_ID = "Qwen/Qwen3.5-0.8B"
MODEL_REVISION = "2fc06364715b967f1860aea9cf38778875588b17"
TRANSFORMERS_VERSION = "5.24.0"

# Original, non-repeated example prose; replace with several independent real
# documents for a research result. No dataset download is needed for this text.
EXAMPLE_TEXT = """On a quiet morning, a researcher arrived at a small public
library to study how people used the building. The entrance faced a busy street,
while the reading room overlooked a garden. Visitors stopped near the door to
check a notice board, then moved toward different sections. Some looked for a
particular book. Others chose a table first and unpacked a laptop or a notebook.
The researcher drew a simple floor plan and marked the routes people took.

The first observation seemed straightforward: most visitors walked past the
information desk without asking for help. It would have been easy to conclude
that the desk was unnecessary. But the researcher also recorded the kinds of
questions that were asked. A few involved books that had been moved, equipment
that needed repair, or resources that could not be found in the catalogue. The
number of interactions alone did not capture their importance. A short
conversation could save someone a long and frustrating search.

Later, a librarian explained how the shelves had changed over the years. The
building had once contained many more printed reference volumes. Some of that
space now held materials for children, language learners, and community groups.
The collection was not shrinking in a simple uniform way. Different sections
changed at different rates as the needs of the neighbourhood evolved. An
inventory counted the objects, but it could not by itself describe how useful
each part of the collection had become.

At noon, sunlight reached the tables by the windows. Several visitors moved
their chairs to avoid glare on their screens. This movement had little to do
with the books or the seating plan. It depended on time, weather, and the angle
of the windows. The researcher added these conditions to the notes. A pattern
that looked permanent during one hour could disappear later in the day. The
same caution applied to observations made during school holidays or examination
weeks, when the mix of visitors was different.

The next task was to compare possible changes to the room. Moving a shelf might
make one path shorter while making another path more confusing. Adding signs
might help first-time visitors but clutter the view for regular users. The staff
agreed to test a small change before committing to a larger renovation. They
would place a temporary sign at one junction, record the questions visitors
asked, and compare the results on similar days. This would not answer every
question about the building, but it would help distinguish a useful idea from
an attractive guess.

Before leaving, the researcher checked the notes for gaps. Several visits had
been interrupted, and a few routes were uncertain because two people crossed
at the same moment. Instead of filling in those details from memory, the
researcher marked them as unknown. The final report would explain what was
measured, what had been inferred, and which conclusions required more evidence.
The librarian suggested repeating the observation in a different season. Both
agreed that a careful first study should create better questions for the next
one, as well as describe what had happened that day.
"""


def matrix64(state):
    """Validate a single state with orientation [key channel, value channel]."""
    x = np.asarray(state, dtype=np.float64)
    if x.ndim != 2 or min(x.shape) < 1 or not np.isfinite(x).all():
        raise ValueError("Expected one finite, nonempty [K, V] state matrix.")
    return x


def spectrum_and_metrics(state):
    """Uncentered SVD: energy means squared singular values, not covariance."""
    x = matrix64(state)
    u, sigma, vh = np.linalg.svd(x, full_matrices=False)
    energy = sigma * sigma
    total = float(energy.sum())
    if total == 0:
        retained = np.zeros_like(energy)
        row_rms = np.zeros(x.shape[0])
        col_rms = np.zeros(x.shape[1])
        metrics = dict(E1=0.0, E4=0.0, E8=0.0, r90=0, stable_rank=0.0,
                       max_over_rms=0.0, top1pct_energy=0.0,
                       u1_ipr=float("nan"), v1_ipr=float("nan"),
                       leading_gap=float("nan"), row_rms_max_over_median=0.0,
                       col_rms_max_over_median=0.0)
    else:
        retained = np.cumsum(energy) / total
        row_rms = np.sqrt(np.mean(x * x, axis=1))
        col_rms = np.sqrt(np.mean(x * x, axis=0))
        entry_energy = np.sort((x * x).ravel())
        topn = max(1, int(math.ceil(0.01 * x.size)))
        metrics = {
            "E1": float(retained[0]),
            "E4": float(retained[min(4, len(retained)) - 1]),
            "E8": float(retained[min(8, len(retained)) - 1]),
            "r90": int(np.searchsorted(retained, 0.9) + 1),
            "stable_rank": float(total / energy[0]),
            "max_over_rms": float(np.max(np.abs(x)) / np.sqrt(total / x.size)),
            "top1pct_energy": float(entry_energy[-topn:].sum() / total),
            "u1_ipr": float(np.sum(u[:, 0] ** 4)),
            "v1_ipr": float(np.sum(vh[0] ** 4)),
            "leading_gap": float((sigma[0] - (sigma[1] if len(sigma) > 1 else 0)) / sigma[0]),
            "row_rms_max_over_median": float(row_rms.max() / (np.median(row_rms) + 1e-12)),
            "col_rms_max_over_median": float(col_rms.max() / (np.median(col_rms) + 1e-12)),
        }
    return metrics, retained, row_rms, col_rms


def fake_quantize(state, bits=4, grouping="tensor"):
    """Affine uniform round-to-nearest, with one min/scale pair per group.

    row: one pair for each K coordinate, reducing across V.
    column: one pair for each V coordinate, reducing across K.
    Returned floats simulate quantization; they are NOT packed INT4/INT8.
    """
    x = matrix64(state)
    if not isinstance(bits, (int, np.integer)) or not 2 <= bits <= 16:
        raise ValueError("bits must be an integer between 2 and 16")
    reductions = {"tensor": None, "row": 1, "column": 0}
    if grouping not in reductions:
        raise ValueError("grouping must be tensor, row, or column")
    axis = reductions[grouping]
    lo = np.min(x, axis=axis, keepdims=True)
    hi = np.max(x, axis=axis, keepdims=True)
    scale = (hi - lo) / ((1 << int(bits)) - 1)
    scale_safe = np.where(scale == 0, 1.0, scale)
    codes = np.clip(np.rint((x - lo) / scale_safe), 0, (1 << int(bits)) - 1)
    reconstructed = codes * scale_safe + lo
    n_groups = 1 if grouping == "tensor" else x.shape[0 if grouping == "row" else 1]
    return reconstructed, {
        "groups": int(n_groups),
        # Hypothetical packed payload + FP32 min + FP32 scale. No packing done.
        "ideal_bits_per_entry_with_metadata": float(bits + 64 * n_groups / x.size),
    }


def relative_frobenius_error(reference, candidate):
    x, y = matrix64(reference), matrix64(candidate)
    if x.shape != y.shape:
        raise ValueError("Shape mismatch")
    return float(np.linalg.norm(y - x, "fro") / (np.linalg.norm(x, "fro") + 1e-12))


def get_cache_state(cache, layer_idx):
    """Transformers v5.24.0 public cache API; no guessed axis transposition."""
    if not hasattr(cache, "get_recurrent_state"):
        raise RuntimeError("Unexpected cache API. Run the pinned transformers==5.24.0 install cell and restart.")
    state = cache.get_recurrent_state(int(layer_idx), state_idx=0)
    if state is None:
        raise RuntimeError(f"Layer {layer_idx} has no recurrent state; select a linear_attention layer.")
    return state


def load_pretrained():
    """Load pretrained weights, retaining their ordinary FP16/BF16 precision."""
    import importlib.util
    optional_kernels = [name for name in ("fla", "causal_conv1d")
                        if importlib.util.find_spec(name) is not None]
    if optional_kernels:
        raise RuntimeError("This experiment requires native PyTorch kernels. Found "
                           + ", ".join(optional_kernels)
                           + ". Use a fresh Colab runtime without flash-linear-attention/causal-conv1d. "
                           "use_kernels=False alone does not disable installed local fallbacks.")
    import torch
    import transformers
    from transformers import AutoTokenizer, Qwen3_5ForConditionalGeneration

    if transformers.__version__ != TRANSFORMERS_VERSION:
        raise RuntimeError(f"Expected transformers {TRANSFORMERS_VERSION}; got {transformers.__version__}. Restart after installing.")
    if not torch.cuda.is_available():
        raise RuntimeError("GPU required for this notebook. In Colab: Runtime > Change runtime type > GPU.")
    torch.manual_seed(17)
    np.random.seed(17)
    # Avoid TF32 changing FP32 state calculations. This is a numeric diagnostic,
    # not a speed benchmark. Do not wrap the forward pass in autocast.
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    try:
        native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
    except TypeError:  # Older Colab PyTorch without the keyword.
        native_bf16 = torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if native_bf16 else torch.float16
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False)
    model = Qwen3_5ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        trust_remote_code=False,
        dtype=dtype,
        device_map={"": "cuda:0"},
        attn_implementation="sdpa",
        use_kernels=False,
    ).eval()
    # The native text backbone returns the same decoder caches and avoids the
    # vision tower and vocabulary projection for this text-only state study.
    backbone = model.model.language_model
    config = model.config.text_config
    if config.model_type != "qwen3_5_text":
        raise RuntimeError(f"Unexpected text architecture: {config.model_type}")
    print("Model:", MODEL_ID, "revision:", MODEL_REVISION)
    print("GPU:", torch.cuda.get_device_name(0), "weight dtype:", dtype)
    print("Linear layers:", [i for i, kind in enumerate(config.layer_types) if kind == "linear_attention"])
    return model, backbone, tokenizer


def write_csv(path, rows):
    if not rows:
        raise ValueError("Cannot write an empty table")
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def analyze_snapshots(snapshots, output_dir="qwen_state_results", bits=(8, 4)):
    """CPU-only snapshot measurements. No model outputs or feedback simulated."""
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    metrics_rows, quant_rows = [], []
    for (tokens, layer, head), state in sorted(snapshots.items()):
        identity = {"tokens": tokens, "layer": layer, "head": head}
        measurements, _, _, _ = spectrum_and_metrics(state)
        metrics_rows.append({**identity, **measurements})
        for b in bits:
            for grouping in ("tensor", "row", "column"):
                reconstructed, storage = fake_quantize(state, int(b), grouping)
                quant_rows.append({**identity, "bits": int(b), "grouping": grouping,
                    "relative_frobenius_error": relative_frobenius_error(state, reconstructed),
                    **storage})
    write_csv(output / "state_metrics.csv", metrics_rows)
    write_csv(output / "static_quantization.csv", quant_rows)
    return metrics_rows, quant_rows


def make_figures(snapshots, metrics_rows, quant_rows, output_dir="qwen_state_results",
                 plot_head=0, label="Pretrained Qwen3.5-0.8B"):
    """Create PNG+PDF figures. A label is required for any synthetic fixture."""
    import matplotlib.pyplot as plt

    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    latest = max(t for t, _, _ in snapshots)
    layers = sorted({layer for t, layer, h in snapshots if t == latest and h == plot_head})
    if not layers:
        raise ValueError("plot_head is not present in the snapshots")
    plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                         "figure.dpi": 140, "savefig.dpi": 200})
    paths = []

    def save(fig, stem):
        fig.savefig(output / (stem + ".png"), bbox_inches="tight")
        fig.savefig(output / (stem + ".pdf"), bbox_inches="tight")
        plt.close(fig)
        paths.append(str(output / (stem + ".png")))

    fig, axes = plt.subplots(len(layers), 3, figsize=(14, 3.6 * len(layers)), squeeze=False,
                             constrained_layout=True)
    for row, layer in enumerate(layers):
        state = snapshots[(latest, layer, plot_head)]
        measures, retained, rows, cols = spectrum_and_metrics(state)
        # Linear absolute magnitude, separate colorbars. Differences in color
        # across layers do not imply equal absolute scales.
        im = axes[row, 0].imshow(np.abs(state), origin="lower", aspect="auto", cmap="magma")
        fig.colorbar(im, ax=axes[row, 0], fraction=0.046, pad=0.04, label="|State entry|")
        axes[row, 0].set(xlabel="Value channel (column)", ylabel="Key channel (row)",
                         title=f"L{layer}, H{plot_head}, t={latest}: magnitude")
        axes[row, 1].plot(rows, label="Row RMS: key channel", color="#0072B2")
        axes[row, 1].plot(cols, label="Column RMS: value channel", color="#D55E00", alpha=.85)
        axes[row, 1].set(xlabel="Channel index", ylabel="RMS", title="Channel scale")
        axes[row, 1].legend(fontsize=8)
        axes[row, 2].plot(np.arange(1, len(retained) + 1), retained, color="#009E73", lw=2)
        axes[row, 2].axhline(.9, ls="--", color="#777777", lw=1)
        axes[row, 2].set(xlabel="Retained rank r", ylabel="Sum of top-r squared singular values / total",
                         ylim=(0, 1.02), title=f"Uncentered SVD: E8={measures['E8']:.3f}; r90={measures['r90']}")
    fig.suptitle(label + " | final checkpoints | color scales differ by layer", fontsize=14)
    save(fig, "state_shape_and_spectrum")

    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    for layer in layers:
        series = sorted((r for r in metrics_rows if r["layer"] == layer and r["head"] == plot_head),
                        key=lambda r: r["tokens"])
        ts = [r["tokens"] for r in series]
        axes[0, 0].plot(ts, [r["E8"] for r in series], marker="o", label=f"L{layer}")
        axes[0, 1].plot(ts, [r["max_over_rms"] for r in series], marker="o", label=f"L{layer}")
        axes[1, 0].plot(ts, [r["u1_ipr"] for r in series], marker="o", label=f"L{layer}")
        axes[1, 1].plot(ts, [r["v1_ipr"] for r in series], marker="o", label=f"L{layer}")
    titles = [("Spectral concentration", "Energy retained at rank 8"),
              ("Entry magnitude concentration", "max |S| / RMS(S)"),
              ("Leading left singular vector", "IPR(u1): key coordinates"),
              ("Leading right singular vector", "IPR(v1): value coordinates")]
    for ax, (title, ylabel) in zip(axes.ravel(), titles):
        ax.set(title=title, xlabel="Tokens processed", ylabel=ylabel)
        ax.legend(fontsize=8)
        ax.grid(alpha=.2)
    fig.suptitle(label + f" | head {plot_head} | one document trajectory", fontsize=14)
    save(fig, "state_time_and_localization")

    fig, axes = plt.subplots(1, len(layers), figsize=(4.6 * len(layers), 4.3), squeeze=False,
                             constrained_layout=True)
    groups, colors = ("tensor", "row", "column"), ("#666666", "#0072B2", "#D55E00")
    precision = sorted({r["bits"] for r in quant_rows}, reverse=True)
    for ax, layer in zip(axes[0], layers):
        for j, (group, color) in enumerate(zip(groups, colors)):
            vals = [next(r["relative_frobenius_error"] for r in quant_rows
                         if r["tokens"] == latest and r["layer"] == layer and r["head"] == plot_head
                         and r["bits"] == bits and r["grouping"] == group) for bits in precision]
            ax.bar(np.arange(len(precision)) + (j - 1) * .24, vals, width=.24, label=group, color=color)
        ax.set(xticks=np.arange(len(precision)), xticklabels=[str(b) + " bit" for b in precision],
               ylabel="Relative Frobenius reconstruction error", title=f"L{layer}, H{plot_head}, t={latest}")
        ax.legend(fontsize=8)
    fig.suptitle(label + " | static fake quantization | same bits, different metadata budgets", fontsize=13)
    save(fig, "static_quantization_axes")
    return paths


In [ ]:
%%writefile state_trajectory.py
"""Actual per-token Qwen3.5 native cache collection (Transformers 5.24.0)."""
from __future__ import annotations

import hashlib
import inspect
import json
import platform
import time
from pathlib import Path
import numpy as np


def _selection(config, layers, heads):
    layers, heads = tuple(map(int, layers)), tuple(map(int, heads))
    if not layers or len(set(layers)) != len(layers):
        raise ValueError("Choose nonempty, distinct layer IDs.")
    if not heads or len(set(heads)) != len(heads):
        raise ValueError("Choose nonempty, distinct head IDs.")
    for layer in layers:
        if not 0 <= layer < len(config.layer_types) or config.layer_types[layer] != "linear_attention":
            raise ValueError(f"Layer {layer} is not linear_attention.")
    if min(heads) < 0 or max(heads) >= config.linear_num_value_heads:
        raise ValueError("Head is outside the linear value-head range.")
    shape = (1, config.linear_num_value_heads, config.linear_key_head_dim, config.linear_value_head_dim)
    if shape[-2:] != (128, 128):
        raise ValueError(f"Expected 128 x 128 state, got {shape[-2:]}.")
    return layers, heads, shape


def _read_selected(cache, layers, head_index, expected_shape):
    import torch
    selected = []
    for layer in layers:
        state = cache.get_recurrent_state(layer, state_idx=0)
        if state is None or tuple(state.shape) != expected_shape:
            raise RuntimeError(f"Layer {layer}: unexpected native state shape {getattr(state, 'shape', None)}")
        # Check BEFORE any conversion: casting BF16 to FP32 would not be an FP32 reference.
        if state.dtype != torch.float32:
            raise RuntimeError(f"Layer {layer}: native state must be FP32, got {state.dtype}")
        selected.append(state[0].index_select(0, head_index))
    # One synchronous transfer per token; no GPU history, no retained cache views.
    return torch.stack(selected).detach().cpu().numpy()


def _check_length(cache, t, config):
    if cache is None or int(cache.get_seq_length()) != t:
        raise RuntimeError(f"Cache sequence length does not equal {t}.")
    for i, kind in enumerate(config.layer_types):
        if kind == "full_attention" and int(cache.get_seq_length(i)) != t:
            raise RuntimeError(f"Attention KV cache length mismatch at layer {i}, token {t}.")


def collect_trajectory(backbone, tokenizer, text, *, max_tokens=256,
                       layers=(0, 8, 16), heads=(0, 1), cpu_budget_mib=1024,
                       progress_every=32, model_id="Qwen/Qwen3.5-0.8B",
                       model_revision="2fc06364715b967f1860aea9cf38778875588b17"):
    """Teacher-forced one-token steps from an empty cache; all selected states."""
    import torch
    import transformers
    if transformers.__version__ != "5.24.0":
        raise RuntimeError("Install transformers==5.24.0 and restart the runtime.")
    if not text.strip() or int(max_tokens) < 1:
        raise ValueError("Use nonempty text and max_tokens >= 1.")
    config = backbone.config
    layers, heads, expected = _selection(config, layers, heads)
    ids_cpu = tokenizer(text, return_tensors="pt", add_special_tokens=False,
                        truncation=True, max_length=int(max_tokens))["input_ids"]
    if ids_cpu.ndim != 2 or ids_cpu.shape[0] != 1 or ids_cpu.shape[1] == 0:
        raise ValueError("Exactly one unpadded, nonempty document is required.")
    T = int(ids_cpu.shape[1])
    shape = (T, len(layers), len(heads), 128, 128)
    nbytes = int(np.prod(shape)) * 4
    if nbytes > cpu_budget_mib * 2**20:
        raise MemoryError(f"Trajectory needs {nbytes/2**20:.1f} MiB; reduce tokens/layers/heads or raise cpu_budget_mib.")
    device = next(backbone.parameters()).device
    ids = ids_cpu.to(device)
    head_index = torch.tensor(heads, device=device, dtype=torch.long)
    states = np.empty(shape, dtype=np.float32)
    cache = None
    gpu = device.type == "cuda"
    backbone.eval()
    if gpu:
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()
    first_hash = None
    print(f"Collecting {T} tokens, {len(layers)*len(heads)} pairs; CPU states {nbytes/2**20:.1f} MiB.")
    with torch.inference_mode():
        for t in range(T):
            # Send ONLY the new token and reuse the SAME hybrid cache.
            # TextModel derives absolute position_ids from the full-attention cache.
            result = backbone(input_ids=ids[:, t:t+1], cache=cache, use_cache=True,
                              output_attentions=False, output_hidden_states=False)
            cache = result.cache
            _check_length(cache, t + 1, config)
            snapshot = _read_selected(cache, layers, head_index, expected)
            if not np.isfinite(snapshot).all():
                raise FloatingPointError(f"Nonfinite state at token {t+1}.")
            states[t] = snapshot  # independent copy into preallocated CPU memory
            if t == 0:
                first_hash = hashlib.sha256(states[0].tobytes()).hexdigest()
            del result, snapshot
            if progress_every and ((t+1) % progress_every == 0 or t+1 == T):
                print(f"  saved {t+1}/{T}")
    if gpu:
        torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - started
    if hashlib.sha256(states[0].tobytes()).hexdigest() != first_hash:
        raise AssertionError("An earlier snapshot was overwritten.")
    token_ids = ids_cpu[0].numpy().astype(np.int64, copy=True)
    token_texts = np.asarray([tokenizer.decode([int(x)], skip_special_tokens=False) for x in token_ids], dtype=np.str_)
    source_path = inspect.getsourcefile(type(backbone))
    meta = {
        "schema_version": 1, "experiment": "Actual per-token recurrent state trajectory",
        "model_id": model_id, "model_revision": model_revision,
        "transformers_version": transformers.__version__, "torch_version": torch.__version__,
        "python_version": platform.python_version(),
        "modeling_source_sha256": hashlib.sha256(Path(source_path).read_bytes()).hexdigest() if source_path else None,
        "device": str(device), "gpu": torch.cuda.get_device_name(device) if gpu else None,
        "weight_dtype": str(next(backbone.parameters()).dtype), "native_state_dtype": "torch.float32",
        "state_axes": ["time", "layer", "head", "key_channel", "value_channel"],
        "state_shape": list(shape), "cache_state_shape": list(expected),
        "layers": list(layers), "heads": list(heads), "tokens_processed": T,
        "forward_tokens_per_call": 1, "sampling_stride": 1,
        "token_semantics": "states[t-1] is the state AFTER processing token_ids[t-1], positions start at 1",
        "text_sha256": hashlib.sha256(text.encode()).hexdigest(),
        "token_ids_sha256": hashlib.sha256(token_ids.tobytes()).hexdigest(),
        "elapsed_collection_seconds": elapsed, "tokens_per_second_including_copy": T / max(elapsed, 1e-12),
        "cpu_state_mib": nbytes / 2**20,
        "gpu_peak_allocated_gib": torch.cuda.max_memory_allocated(device) / 2**30 if gpu else None,
        "gpu_peak_reserved_gib": torch.cuda.max_memory_reserved(device) / 2**30 if gpu else None,
        "first_snapshot_immutable": True,
        "validation_status": "not_run",
        "limitations": ["FP16/BF16 weights and activations with native FP32 state, not an all-FP32 model.",
                        "Teacher-forced document tokens; no generation, no padding, no repeated text.",
                        "Single-token and chunked mixed-precision execution can differ numerically.",
                        "GPU memory includes weights/current cache/temporaries; CPU history is separate.",
                        "State probes and transfers are included in speed; not a model throughput benchmark."]}
    del cache, ids, head_index
    return {"states": states, "layer_ids": np.asarray(layers, dtype=np.int64),
            "head_ids": np.asarray(heads, dtype=np.int64), "token_indices": np.arange(1, T+1, dtype=np.int64),
            "token_ids": token_ids, "token_texts": token_texts, "metadata": meta}


def validate_trajectory(backbone, data, *, replay_tokens=32, prefix_tokens=None,
                        replay_rtol=1e-5, replay_atol=1e-7, prefix_rtol=None, prefix_atol=1e-5):
    """Fresh-cache replay + independently chunked prefix comparison. Never crop cache."""
    import torch
    if int(replay_tokens) < 1:
        raise ValueError("replay_tokens must be >= 1; an empty replay is not validation.")
    config = backbone.config
    layers, heads, expected = _selection(config, data["layer_ids"], data["head_ids"])
    states = data["states"]
    T = len(data["token_ids"])
    if states.dtype != np.float32 or states.shape != (T, len(layers), len(heads), 128, 128):
        raise AssertionError("Trajectory dtype/shape mismatch.")
    # Bound temporary allocations rather than isfinite() on the entire history.
    for frame in states:
        if not np.isfinite(frame).all():
            raise AssertionError("Nonfinite trajectory.")
    np.testing.assert_array_equal(data["token_indices"], np.arange(1, T+1))
    device = next(backbone.parameters()).device
    weight_dtype = next(backbone.parameters()).dtype
    if prefix_rtol is None:
        prefix_rtol = .05 if weight_dtype == torch.bfloat16 else (.01 if weight_dtype == torch.float16 else .0002)
    for tolerance in (replay_rtol, replay_atol, prefix_rtol, prefix_atol):
        if not np.isfinite(tolerance) or tolerance < 0:
            raise ValueError("Validation tolerances must be finite and nonnegative.")
    ids = torch.as_tensor(data["token_ids"], device=device, dtype=torch.long)[None]
    head_index = torch.tensor(heads, device=device)
    rows = []
    started = time.perf_counter()

    def compare(observed, reference, t, method, rtol, atol, elementwise=False):
        for li, layer in enumerate(layers):
            for hi, head in enumerate(heads):
                actual = observed[li, hi].astype(np.float64)
                target = reference[li, hi].astype(np.float64)
                diff = actual - target
                rms = float(np.sqrt(np.mean(target * target)))
                rmse = float(np.sqrt(np.mean(diff * diff)))
                passed = bool(np.allclose(actual, target, rtol=rtol, atol=atol)) if elementwise else rmse <= atol + rtol*rms
                rows.append(dict(method=method, token=t, layer=layer, head=head,
                                 max_abs_error=float(np.max(np.abs(diff))), rmse=rmse,
                                 relative_frobenius=rmse/max(rms, 1e-30),
                                 rtol=rtol, atol=atol, criterion="elementwise_allclose" if elementwise else "rmse <= atol + rtol * reference_rms",
                                 passed=passed))

    backbone.eval()
    with torch.inference_mode():
        cache = None
        for t in range(min(int(replay_tokens), T)):
            result = backbone(input_ids=ids[:, t:t+1], cache=cache, use_cache=True)
            cache = result.cache
            _check_length(cache, t+1, config)
            compare(_read_selected(cache, layers, head_index, expected), states[t], t+1,
                    "independent_one_token_replay", replay_rtol, replay_atol, True)
            del result
        del cache
        checkpoints = sorted(set(int(x) for x in (prefix_tokens or [1, min(8,T), min(32,T), T])))
        for t in checkpoints:
            if not 1 <= t <= T:
                raise ValueError("Validation prefix out of bounds.")
            result = backbone(input_ids=ids[:, :t], cache=None, use_cache=True)
            _check_length(result.cache, t, config)
            full = _read_selected(result.cache, layers, head_index, expected)
            compare(full, states[t-1], t, "fresh_whole_prefix_vs_trajectory", prefix_rtol, prefix_atol)
            del result
            # Different segmentation explicitly exercises cached multi-token conv/GDN continuation.
            if t > 2:
                cache = None
                cuts = sorted(set([0, min(3,t), min(8,t), t]))
                for a,b in zip(cuts[:-1], cuts[1:]):
                    result = backbone(input_ids=ids[:, a:b], cache=cache, use_cache=True)
                    cache = result.cache
                    _check_length(cache, b, config)
                    del result
                compare(_read_selected(cache, layers, head_index, expected), full, t,
                        "segmented_prefix_vs_whole_prefix", prefix_rtol, prefix_atol)
                del cache
    replay_ok = all(r["passed"] for r in rows if r["method"] == "independent_one_token_replay")
    prefix_ok = all(r["passed"] for r in rows if r["method"] != "independent_one_token_replay")
    report = {"schema_shape_finite": True, "replay_passed": replay_ok, "prefix_passed": prefix_ok,
              "status": "passed" if replay_ok and prefix_ok else "needs_review",
              "replay_tokens": min(int(replay_tokens), T), "prefix_tokens": checkpoints,
              "elapsed_seconds": time.perf_counter()-started, "rows": rows,
              "note": "Prefix tolerances are numerical smoke checks, not a proof of bitwise equivalence or model accuracy."}
    data["metadata"]["validation_status"] = report["status"]
    data["metadata"]["validation"] = {k:v for k,v in report.items() if k != "rows"}
    if not replay_ok or not prefix_ok:
        print("WARNING: validation needs review. Inspect failed rows; do not label these results validated.")
    else:
        print(f"Validation passed ({len(rows)} comparisons).")
    return report


def export_trajectory(data, output_dir, *, text=None, validation=None, verify_roundtrip=True):
    """Portable, pickle-free NPZ. Roundtrip temporarily loads one extra states array."""
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    payload = {k: data[k] for k in ("states", "layer_ids", "head_ids", "token_indices", "token_ids", "token_texts")}
    payload["metadata_json"] = np.asarray(json.dumps(data["metadata"], ensure_ascii=False))
    path = output / "recurrent_state_trajectory.npz"
    np.savez_compressed(path, **payload)
    if verify_roundtrip:
        with np.load(path, allow_pickle=False) as loaded:
            for key, value in payload.items():
                np.testing.assert_array_equal(loaded[key], value, err_msg=f"NPZ roundtrip: {key}")
        print("NPZ exact roundtrip passed (allow_pickle=False).")
    (output / "run_metadata.json").write_text(json.dumps(data["metadata"], ensure_ascii=False, indent=2), encoding="utf-8")
    if validation is not None:
        (output / "validation_report.json").write_text(json.dumps(validation, indent=2), encoding="utf-8")
    if text is not None:
        (output / "input_document.txt").write_text(text, encoding="utf-8")
    print(f"Saved {path}: {path.stat().st_size/2**20:.1f} MiB compressed.")
    return path


def load_trajectory(path):
    with np.load(path, allow_pickle=False) as z:
        data = {key:z[key] for key in ("states", "layer_ids", "head_ids", "token_indices", "token_ids", "token_texts")}
        data["metadata"] = json.loads(str(z["metadata_json"].item()))
    expected = (len(data["token_ids"]), len(data["layer_ids"]), len(data["head_ids"]), 128, 128)
    if data["states"].dtype != np.float32 or data["states"].shape != expected:
        raise ValueError("Invalid FP32 trajectory schema.")
    np.testing.assert_array_equal(data["token_indices"], np.arange(1, expected[0]+1))
    return data


def checkpoint_snapshots(data, checkpoint_tokens=(64, 128, 192, 256)):
    """Views of already collected states for original spectrum/quantization experiments."""
    T = len(data["states"])
    checkpoints = sorted({int(t) for t in checkpoint_tokens if 1 <= t <= T} | {T})
    return {(t, int(layer), int(head)): data["states"][t-1, li, hi]
            for t in checkpoints for li, layer in enumerate(data["layer_ids"])
            for hi, head in enumerate(data["head_ids"])}


In [ ]:
%%writefile state_viewer.py
"""Interactive CPU-side views of actual FP32 recurrent-state trajectories.

No model execution or invented intermediate states occur in this module.
"""
from __future__ import annotations

import base64
import html
import json
from pathlib import Path

import numpy as np


def validate_trajectory(data):
    """Validate the portable NPZ schema without copying the full state tensor."""
    required = ("states", "layer_ids", "head_ids", "token_indices", "token_ids", "token_texts")
    missing = [key for key in required if key not in data]
    if missing:
        raise ValueError(f"Missing trajectory fields: {missing}")
    states = np.asarray(data["states"])
    if states.dtype != np.float32 or states.ndim != 5 or states.shape[-2:] != (128, 128):
        raise ValueError("states must be FP32 [T, L, H, 128, 128].")
    if min(states.shape[:3]) < 1:
        raise ValueError("The trajectory must contain at least one time, layer, and head.")
    vectors = {key: np.asarray(data[key]) for key in required[1:]}
    for key, expected in (("layer_ids", states.shape[1]), ("head_ids", states.shape[2]),
                          ("token_indices", states.shape[0]), ("token_ids", states.shape[0]),
                          ("token_texts", states.shape[0])):
        if vectors[key].ndim != 1 or len(vectors[key]) != expected:
            raise ValueError(f"{key} must be a 1D vector of length {expected}.")
    for key in ("layer_ids", "head_ids", "token_indices", "token_ids"):
        if vectors[key].dtype.kind not in "iu":
            raise ValueError(f"{key} must contain integers.")
    for key in ("layer_ids", "head_ids"):
        if len(np.unique(vectors[key])) != len(vectors[key]):
            raise ValueError(f"Duplicate {key} are not allowed.")
    if not np.array_equal(vectors["token_indices"], np.arange(1, len(states) + 1)):
        raise ValueError("Expected every token exactly once, indexed from 1 through T.")
    if vectors["token_texts"].dtype.kind not in "US":
        raise ValueError("token_texts must be a Unicode/string array, never an object array.")
    for i, frame in enumerate(states):
        if not np.isfinite(frame).all():
            raise ValueError(f"Nonfinite recurrent state at token {i + 1}.")
    return {"states": states, **vectors}


def load_trajectory(path):
    """Read only numeric/string NPZ fields; pickle is deliberately disabled."""
    with np.load(path, allow_pickle=False) as archive:
        data = {key: archive[key] for key in archive.files}
    validate_trajectory(data)
    return data


def _limits(states):
    # No large abs(states) allocation: extrema have the same maximum magnitude.
    return np.maximum(states.max(axis=(0, 3, 4)), -states.min(axis=(0, 3, 4)))


def build_explorer(data):
    """Return a live Colab/Jupyter widget, transferring one surface per update.

    Install plotly>=6,<7, anywidget, ipywidgets first. In Colab call
    google.colab.output.enable_custom_widget_manager() before displaying it.
    Keep the returned widget and call close_explorer(widget) before replacing it.
    """
    import ipywidgets as widgets
    import plotly.graph_objects as go

    d = validate_trajectory(data)
    states = d["states"]
    n = len(states)
    pair_max = _limits(states)
    global_max = float(pair_max.max())
    layer = widgets.Dropdown(options=[(f"Layer {int(v)}", i) for i, v in enumerate(d["layer_ids"])],
                             description="Layer", layout=widgets.Layout(width="180px"))
    head = widgets.Dropdown(options=[(f"Head {int(v)}", i) for i, v in enumerate(d["head_ids"])],
                            description="Head", layout=widgets.Layout(width="180px"))
    mode = widgets.Dropdown(options=[("Signed S", "signed"), ("Magnitude |S|", "magnitude")],
                            description="Height", layout=widgets.Layout(width="230px"))
    scope = widgets.Dropdown(options=[("This pair / all times", "pair"),
                                      ("All layers + heads + times", "global")],
                             description="Z scale", layout=widgets.Layout(width="310px"))
    stride = widgets.Dropdown(options=[("128 × 128 (full)", 1), ("64 × 64 (view only)", 2),
                                       ("32 × 32 (view only)", 4)],
                              description="Grid", layout=widgets.Layout(width="250px"))
    time_slider = widgets.IntSlider(value=1, min=1, max=n, step=1, description="Token t",
                                    continuous_update=False,
                                    layout=widgets.Layout(width="75%"))
    play = widgets.Play(value=1, min=1, max=n, step=1, interval=900, repeat=False,
                        description="Play", show_repeat=True)
    link = widgets.jslink((play, "value"), (time_slider, "value"))
    speed = widgets.Dropdown(options=[("Slow · 1.5 s", 1500), ("Normal · 0.9 s", 900),
                                      ("Fast · 0.4 s", 400)], value=900,
                             description="Playback", layout=widgets.Layout(width="240px"))
    reset = widgets.Button(description="Reset camera", icon="refresh")
    status = widgets.HTML()
    note = widgets.HTML(value=("<small>드래그: 회전 · 휠/핀치: 확대·축소. "
                               "Z축과 색 범위는 선택한 범위의 모든 시간에 대해 고정됩니다. "
                               "Grid는 표시 해상도만 변경하며 저장 데이터와 통계는 항상 128×128입니다.</small>"))
    fig = go.FigureWidget(data=[go.Surface(z=states[0, 0, 0],
        x=np.arange(128), y=np.arange(128), colorscale="RdBu_r", showscale=True,
        hovertemplate="Value channel=%{x}<br>Key channel=%{y}<br>Height=%{z:.6g}<extra></extra>")])
    camera = dict(eye=dict(x=1.5, y=1.5, z=1.15))
    fig.update_layout(height=610, margin=dict(l=0, r=0, b=15, t=45),
                      template="plotly_white", uirevision="recurrent-state-camera",
                      scene=dict(xaxis=dict(title="Value channel", range=[0, 127], autorange=False),
                                 yaxis=dict(title="Key channel", range=[0, 127], autorange=False),
                                 zaxis=dict(title="State S", autorange=False),
                                 aspectmode="manual", aspectratio=dict(x=1, y=1, z=0.7),
                                 dragmode="orbit", camera=camera,
                                 uirevision="recurrent-state-camera"))

    def refresh(change=None):
        i, li, hi = time_slider.value - 1, layer.value, head.value
        signed = mode.value == "signed"
        raw = states[i, li, hi]
        step = stride.value
        sampled = raw[::step, ::step]
        z = sampled if signed else np.abs(sampled)
        bound = max(float(pair_max[li, hi]) if scope.value == "pair" else global_max, 1e-12)
        lower = -bound if signed else 0.0
        with fig.batch_update():
            fig.data[0].x = np.arange(0, 128, step)
            fig.data[0].y = np.arange(0, 128, step)
            fig.data[0].z = z
            fig.data[0].colorscale = "RdBu_r" if signed else "Viridis"
            fig.data[0].cauto = False
            fig.data[0].cmin, fig.data[0].cmax = lower, bound
            fig.data[0].colorbar.title = "S" if signed else "|S|"
            fig.layout.scene.zaxis.range = [lower, bound]
            fig.layout.scene.zaxis.autorange = False
            fig.layout.scene.zaxis.title = "State S" if signed else "Magnitude |S|"
            fig.layout.title = f"State after token {i + 1}/{n} · L{d['layer_ids'][li]} · H{d['head_ids'][hi]}"
        # Float64 on one CPU frame avoids low-precision RMS accumulation.
        maximum = float(max(raw.max(), -raw.min()))
        rms = float(np.sqrt(np.mean(np.square(raw, dtype=np.float64))))
        token = html.escape(str(d["token_texts"][i]))
        status.value = (f"<b>t={i + 1}</b> · token_id={int(d['token_ids'][i])} · "
                        f"token=<code style='white-space:pre-wrap'>{token}</code><br>"
                        f"max |S|={maximum:.6g} · RMS={rms:.6g} · "
                        f"fixed z=[{lower:.6g}, {bound:.6g}] · FP32 128×128")

    for control in (time_slider, layer, head, mode, scope, stride):
        control.observe(refresh, names="value")
    speed.observe(lambda change: setattr(play, "interval", int(change["new"])), names="value")
    reset.on_click(lambda button: fig.update_layout(scene_camera=camera))
    refresh()
    root = widgets.VBox([widgets.HBox([layer, head, mode], layout=widgets.Layout(flex_flow="row wrap")),
                         widgets.HBox([scope, stride], layout=widgets.Layout(flex_flow="row wrap")),
                         widgets.HBox([play, time_slider]),
                         widgets.HBox([speed, reset]), status, fig, note])
    # Hold the link and children for clean disposal when rerunning a cell.
    root._state_explorer_resources = (link, fig, [layer, head, mode, scope, stride,
                                               time_slider, play, speed, reset, status, note])
    return root


def close_explorer(widget):
    """Dispose a previous live explorer before displaying another."""
    resources = getattr(widget, "_state_explorer_resources", None)
    if resources is not None:
        resources[0].unlink()

    def close_tree(node):
        for child in getattr(node, "children", ()):
            close_tree(child)
        node.close()

    close_tree(widget)


def export_pair_html(data, path, *, layer_id=None, head_id=None):
    """Export an offline standalone Plotly viewer for ONE selected layer/head.

    The widget above supports all selected layers/heads. This optional fallback
    embeds one pair to keep browser memory and the HTML download manageable.
    Every token and all 128×128 FP32 values are retained (no frame duplication).
    """
    from plotly.offline import get_plotlyjs

    d = validate_trajectory(data)
    layer_id = int(d["layer_ids"][0]) if layer_id is None else int(layer_id)
    head_id = int(d["head_ids"][0]) if head_id is None else int(head_id)
    ls, hs = d["layer_ids"].tolist(), d["head_ids"].tolist()
    if layer_id not in ls or head_id not in hs:
        raise ValueError("Select layer_id/head_id present in this trajectory.")
    pair = np.ascontiguousarray(d["states"][:, ls.index(layer_id), hs.index(head_id)], dtype="<f4")
    bound = max(float(max(pair.max(), -pair.min())), 1e-12)
    config = dict(n=len(pair), k=128, v=128, layer=layer_id, head=head_id, bound=bound,
                  token_ids=d["token_ids"].tolist(), token_texts=d["token_texts"].astype(str).tolist())
    config_json = json.dumps(config, ensure_ascii=True).replace("<", "\\u003c")
    encoded = base64.b64encode(pair.tobytes()).decode("ascii")
    template = r'''<!doctype html><html lang="en"><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Recurrent State Explorer</title>
<style>body{font-family:system-ui,sans-serif;margin:16px;color:#172033}header{font-size:23px;font-weight:650} .controls{display:flex;gap:14px;flex-wrap:wrap;align-items:center;margin:12px 0}button,select{font:inherit;padding:6px}input[type=range]{width:min(600px,65vw)}#plot{width:100%;height:620px}#info{white-space:pre-wrap;font-family:monospace;font-size:13px}small{display:block;max-width:950px;color:#526075}</style>
<header>Recurrent State Explorer</header>
<div class="controls"><button id="play">▶ Play</button><input id="time" type="range" min="1" max="1" value="1" step="1"><span id="position"></span></div>
<div class="controls"><label>Height <select id="mode"><option value="signed">Signed S</option><option value="magnitude">Magnitude |S|</option></select></label><label>Grid <select id="stride"><option value="1">128 × 128</option><option value="2">64 × 64 (view only)</option><option value="4">32 × 32 (view only)</option></select></label><label>Playback <select id="speed"><option value="900">0.9 s</option><option value="1500">1.5 s</option><option value="400">0.4 s</option></select></label><button id="reset">Reset camera</button></div>
<div id="info"></div><div id="plot"></div>
<small>State immediately after each input token. X: value channel. Y: key channel. Z and color ranges stay fixed across all times. Drag to rotate, scroll/pinch to zoom. This HTML contains one layer/head; use the notebook widget to select among all recorded pairs. Full FP32 matrices are retained; Grid changes only the display.</small>
<script>__PLOTLY_JS__</script>
<script>
const cfg=__CONFIG__;
const bytes=Uint8Array.from(atob('__DATA__'), c=>c.charCodeAt(0));
const states=new Float32Array(bytes.buffer);
const $=id=>document.getElementById(id), plot=$('plot');
$('time').max=cfg.n;
const camera={eye:{x:1.5,y:1.5,z:1.15}};
let initialized=false,busy=false,pending=false,playing=false;
function contents(){
  const t=Number($('time').value)-1,step=Number($('stride').value),signed=$('mode').value==='signed';
  const xy=Array.from({length:Math.ceil(128/step)},(_,i)=>i*step),z=[];
  let max=0,sq=0;
  for(let j=0;j<16384;j++){const s=states[t*16384+j];max=Math.max(max,Math.abs(s));sq+=s*s;}
  for(const k of xy){const row=[];for(const v of xy){const s=states[t*16384+k*128+v];row.push(signed?s:Math.abs(s));}z.push(row);}
  const lower=signed?-cfg.bound:0;
  $('position').textContent=`${t+1} / ${cfg.n}`;
  $('info').textContent=`L${cfg.layer} · H${cfg.head} · after token ${t+1} · token_id=${cfg.token_ids[t]} · token=${JSON.stringify(cfg.token_texts[t])}\nmax |S|=${max.toPrecision(6)} · RMS=${Math.sqrt(sq/16384).toPrecision(6)} · fixed z=[${lower.toPrecision(6)}, ${cfg.bound.toPrecision(6)}]`;
  return {data:[{type:'surface',x:xy,y:xy,z,colorscale:signed?'RdBu':'Viridis',reversescale:signed,cauto:false,cmin:lower,cmax:cfg.bound,colorbar:{title:{text:signed?'S':'|S|'}},hovertemplate:'Value channel=%{x}<br>Key channel=%{y}<br>Height=%{z:.6g}<extra></extra>'}],layout:{margin:{l:0,r:0,b:15,t:40},title:{text:`State after token ${t+1}/${cfg.n} · L${cfg.layer} · H${cfg.head}`},uirevision:'fixed-camera',scene:{xaxis:{title:{text:'Value channel'},range:[0,127],autorange:false},yaxis:{title:{text:'Key channel'},range:[0,127],autorange:false},zaxis:{title:{text:signed?'State S':'Magnitude |S|'},range:[lower,cfg.bound],autorange:false},aspectmode:'manual',aspectratio:{x:1,y:1,z:0.7},dragmode:'orbit',uirevision:'fixed-camera'}}};
}
async function refresh(){
  if(busy){pending=true;return;}
  busy=true;
  try{do{pending=false;const c=contents();if(!initialized){c.layout.scene.camera=camera;await Plotly.newPlot(plot,c.data,c.layout,{responsive:true,scrollZoom:true,displaylogo:false});initialized=true;}else{await Plotly.react(plot,c.data,c.layout,{responsive:true,scrollZoom:true,displaylogo:false});}}while(pending);}
  catch(e){$('info').textContent='Rendering error: '+e;playing=false;$('play').textContent='▶ Play';}
  finally{busy=false;}
}
let playbackEpoch=0;
async function playback(epoch){
  while(playing && epoch===playbackEpoch){
    await new Promise(r=>setTimeout(r,Number($('speed').value)));
    if(!playing || epoch!==playbackEpoch)break;
    if(busy)continue;
    if(Number($('time').value)>=cfg.n){playing=false;$('play').textContent='▶ Play';break;}
    $('time').value=Number($('time').value)+1;await refresh();
  }
}
$('play').onclick=()=>{playing=!playing;$('play').textContent=playing?'❚❚ Pause':'▶ Play';playbackEpoch++;if(playing){if(Number($('time').value)>=cfg.n)$('time').value=1;refresh();playback(playbackEpoch);}};
$('time').oninput=refresh;$('mode').onchange=refresh;$('stride').onchange=refresh;
$('reset').onclick=()=>Plotly.relayout(plot,{'scene.camera':camera});
refresh();
</script></html>'''
    page = template.replace("__PLOTLY_JS__", get_plotlyjs()).replace("__CONFIG__", config_json).replace("__DATA__", encoded)
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(page, encoding="utf-8")
    return target


## 3. 실험 설정

번호는 모두 **0부터 시작**합니다. `LAYERS × HEADS`의 모든 조합을 수집합니다. Qwen3.5-0.8B의 full-attention layer(3, 7, 11, 15, 19, 23)는 recurrent state가 없으므로 선택할 수 없습니다. head는 linear attention의 **value head** 번호 0–15입니다.

입력은 학습된 모델에 실제 문서를 순서대로 넣는 teacher forcing입니다. 생성 토큰을 sampling하지 않으며, padding이나 짧은 문서 반복으로 길이를 채우지 않습니다. 시간축은 실제 토큰 수에서 끝납니다. collection stride는 항상 1입니다.

In [ ]:
import importlib
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Image, FileLink
import recurrent_gpu_probe as probe
import state_trajectory as trajectory_tools
import state_viewer as viewer
probe = importlib.reload(probe)
trajectory_tools = importlib.reload(trajectory_tools)
viewer = importlib.reload(viewer)

MAX_TOKENS = 256
LAYERS = (0, 8, 16)
HEADS = (0, 1)
CPU_STATE_BUDGET_MIB = 1024  # trajectory 배열만의 상한; 저장 검증 시 추가 RAM 필요
CHECKPOINT_TOKENS = (64, 128, 192, 256)  # 기존 실험 1/3 + 시간별 통계용. 수집 간격 아님!
REPLAY_TOKENS = 32          # 정확성 검사: 새 cache로 초기 32토큰을 다시 실행
OUTPUT_DIR = Path("qwen_state_trajectory_results")  # 문서/설정 변경 시 새 폴더명 사용
TEXT = probe.EXAMPLE_TEXT   # 원하는 실제 문서로 교체

estimated_mib = MAX_TOKENS * len(LAYERS) * len(HEADS) * 128 * 128 * 4 / 2**20
print(f"예상 CPU trajectory 상한: {estimated_mib:.1f} MiB")
print("선택 layer/head:", LAYERS, HEADS)
print(TEXT[:350])


## 4. 모델 로드 — 원본 가중치와 실행 경로 유지

- 원본 model revision `2fc06364715b967f1860aea9cf38778875588b17`, Transformers `5.24.0`을 고정합니다.
- native BF16 지원 GPU에서는 BF16, T4처럼 그렇지 않은 GPU에서는 FP16 가중치를 사용합니다. BF16 **에뮬레이션** 지원을 native 지원으로 오인하지 않도록 수정했습니다. 기존 결과가 T4에서 BF16이었다면 새 실행의 FP16 결과와 수치가 달라질 수 있습니다.
- weight 양자화와 학습은 없습니다. TF32를 끄고 `autocast`를 사용하지 않습니다.
- `use_kernels=False`로 Hub 커널을 끄고, 로컬 `fla`/`causal_conv1d` 설치도 확인합니다. 이 옵션 하나만으로 로컬 fallback 커널까지 비활성화되지는 않습니다.
- 멀티모달 checkpoint를 로드하되 텍스트 backbone만 forward합니다. vision tower와 vocabulary 출력 projection은 실행하지 않습니다. 로드된 vision 가중치 메모리는 전체 모델 메모리에 포함됩니다.

In [ ]:
model, backbone, tokenizer = probe.load_pretrained()


## 5. 매 토큰의 실제 recurrent state 수집

첫 forward는 `cache=None`에서 토큰 하나를 처리하여 hybrid cache를 만듭니다. 이후에는 **같은 `result.cache`를 다음 forward의 `cache=`에 전달**하고 아직 처리하지 않은 토큰 하나만 입력합니다. 모델 내부가 convolution state, recurrent state, full-attention KV와 위치를 함께 이어갑니다. `generate()`, prefix 재입력, cache 초기화, cache crop은 수집 루프에서 사용하지 않습니다.

매 forward 직후 `cache.get_recurrent_state(layer, state_idx=0)`에서 원래 state를 읽습니다. native shape `[1, 16, 128, 128]`와 **원래 dtype FP32**를 먼저 검사합니다. 선택한 head만 독립적인 CPU 배열로 복사하므로 다음 in-place cache 갱신에 덮어써지지 않습니다. 모든 원소를 저장하며, energy·SVD·heatmap을 원래 state 대신 저장하지 않습니다.

저장 shape: `[T, L, H, 128, 128]`. `states[t-1, li, hi]`는 **t번째 토큰을 처리한 직후**의 행렬입니다. 행은 key channel, 열은 value channel이며 행렬 내부 축은 토큰 축이 아닙니다. `li/hi`는 선택 목록 안의 위치이고 실제 번호는 `layer_ids/head_ids`로 찾습니다.

FP32는 recurrent 연산과 저장 state에 대한 설명입니다. 모델 전체의 가중치/activation이 FP32라는 뜻은 아닙니다. 첫 토큰은 native chunk 경로로 초기화하고 이후 native recurrent 경로로 이어갑니다. 수학식을 별도 재구현하거나 모델 forward를 monkey patch하지 않습니다.

In [ ]:
trajectory = trajectory_tools.collect_trajectory(
    backbone, tokenizer, TEXT, max_tokens=MAX_TOKENS,
    layers=LAYERS, heads=HEADS, cpu_budget_mib=CPU_STATE_BUDGET_MIB,
    model_id=probe.MODEL_ID, model_revision=probe.MODEL_REVISION,
)
metadata = trajectory["metadata"]
print("states:", trajectory["states"].shape, trajectory["states"].dtype)
print(f"수집 시간: {metadata['elapsed_collection_seconds']:.2f} s")
print(f"수집·복사 포함: {metadata['tokens_per_second_including_copy']:.2f} tokens/s")
print("GPU peak allocated/reserved GiB:",
      metadata["gpu_peak_allocated_gib"], metadata["gpu_peak_reserved_gib"])
print("CPU state MiB:", metadata["cpu_state_mib"])


## 6. 수집 정확성 검사

수집 중 매 토큰마다 native dtype/shape, 유한값, 전체 attention layer의 cache 길이=t를 검사하고 첫 snapshot이 이후에 변하지 않았는지 확인합니다. 아래 검사는 **실제로 수집한 데이터**를 다시 모델 실행과 비교합니다.

1. **독립 재실행:** 새 cache로 처음 `REPLAY_TOKENS`를 다시 한 토큰씩 실행해 매 시점 모든 선택 행렬을 원소별 비교합니다 (`rtol=1e-5`, `atol=1e-7`).
2. **새 prefix:** t=1, 8, 32, 마지막 토큰의 prefix를 각각 빈 cache에서 한 번에 실행하여 저장값과 비교합니다(짧은 문서는 가능한 시점만). 각 비교의 상대 Frobenius 오차와 최대 절댓값 오차를 남깁니다.
3. **다른 분할:** 같은 prefix를 3, 8, 마지막 위치로 나눠 기존 cache를 이어 실행하고 한 번에 처리한 결과와 비교합니다. convolution과 recurrent continuity를 추가 검사합니다.
4. 다음 저장 셀은 NPZ를 `allow_pickle=False`로 다시 읽어 모든 배열을 정확히 비교합니다.

같은 가중치라도 단일 토큰과 여러 토큰의 projection/conv/attention 및 chunk 누적 순서가 달라 mixed precision에서 완전한 bit 일치를 기대하지 않습니다. prefix 비교 통과 기준은 `RMSE ≤ 1e-5 + rtol × reference RMS`이며 기본 rtol은 FP16 1%, BF16 5%, FP32 0.02%입니다. 이는 **명시적인 수치 smoke check**이며 정확성 증명이나 모델 성능 검사가 아닙니다. 아주 작은 state에서는 상대 오차가 커질 수 있으므로 절대 오차도 함께 확인합니다.

하나라도 실패하면 결과에 `needs_review`가 기록됩니다. 오차를 보지 않고 tolerance를 키우지 마십시오. dtype/API/비유한값 오류는 즉시 중단합니다. `cache.crop()`으로 과거 state를 복원하지 않습니다. 검증 시간은 수집 속도 측정에 포함되지 않습니다.

In [ ]:
validation = trajectory_tools.validate_trajectory(
    backbone, trajectory, replay_tokens=REPLAY_TOKENS,
)
validation_df = pd.DataFrame(validation["rows"])
display(validation_df.groupby("method", sort=False).agg(
    comparisons=("passed", "size"), all_passed=("passed", "all"),
    max_abs_error=("max_abs_error", "max"),
    max_relative_frobenius=("relative_frobenius", "max"),
))
if validation["status"] != "passed":
    display(validation_df.loc[~validation_df["passed"]])
print("Validation status:", validation["status"])
print("Replay tokens / prefix checkpoints:", validation["replay_tokens"], validation["prefix_tokens"])


## 7. NPZ export — 전체 FP32 데이터 보존

| 키 | 내용 |
|---|---|
| `states` | FP32 `[T,L,H,128,128]`, 모든 토큰의 모든 선택 행렬 |
| `token_indices` | 1…T, 토큰 처리 후 시점 |
| `token_ids`, `token_texts` | 실제 입력 ID와 단일 토큰 표시 문자열 |
| `layer_ids`, `head_ids` | 배열 index에 대응하는 실제 layer/head 번호 |
| `metadata_json` | revision, 버전, dtype, 축, 입력 hash, 수집 속도/메모리, 검증 상태 |

단일 토큰 decode 문자열은 부분 UTF-8 토큰에서 대체 문자로 보일 수 있습니다. 정확한 입력은 `token_ids`가 기준입니다. NPZ는 객체/pickle을 사용하지 않고 FP32 bit를 그대로 보존합니다. 입력 문서, 검증 상세 보고서와 설정은 별도 파일로도 저장합니다.

In [ ]:
npz_path = trajectory_tools.export_trajectory(
    trajectory, OUTPUT_DIR, text=TEXT, validation=validation,
    verify_roundtrip=True,
)
validation_df.to_csv(OUTPUT_DIR / "validation_comparisons.csv", index=False)
display(FileLink(str(npz_path)))


## 8. 실험 2 — 인터랙티브 Plotly 3D surface

- **Token t:** 원하는 시간으로 이동. ▶/일시정지로 순차 재생. 표시되는 표면은 그 토큰 직후의 실제 state입니다.
- **Layer / Head:** 수집한 layer/head 중 선택. 목록에 없는 조합은 설정을 바꿔 다시 수집해야 합니다.
- **Signed S / Magnitude |S|:** z 높이를 부호 포함 state 또는 절댓값으로 변경합니다. 색도 같은 값을 표시합니다.
- **Z scale:** 기본은 선택한 pair의 **모든 시간**에서 구한 최대 절댓값 M을 사용합니다. signed는 `[-M,M]`, magnitude는 `[0,M]`로 z축과 색 범위를 함께 고정합니다. layer/head 간 절대 크기를 비교하려면 `All layers + heads + times`를 선택합니다. 시점마다 자동 정규화하지 않습니다.
- **회전/줌:** 드래그로 회전, 휠/핀치 또는 Plotly 툴바로 줌. 시간/모드를 바꾸어도 카메라를 유지합니다. Reset camera로 초기화합니다.
- **Grid:** 기본 128×128 전체. 브라우저가 느리면 64×64/32×32 표시에만 간격 샘플링합니다. 저장 데이터·통계·고정 범위 계산은 전체 128×128입니다. 낮은 표시 해상도는 좁은 peak를 놓칠 수 있습니다.

수천 개의 Plotly frame이나 모든 layer/head의 표면을 브라우저에 한꺼번에 복제하지 않고 **현재 표면 하나만 갱신**합니다. 라이브 위젯은 Colab 세션 연결이 유지되어야 하며, 대형 surface는 재생이 목표 간격보다 느릴 수 있습니다. 출력 영역이 접히면 펼치거나 별도 탭의 HTML 보기로 확인하십시오.

In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
if "explorer" in globals():
    viewer.close_explorer(explorer)
explorer = viewer.build_explorer(trajectory)
display(explorer)


### 선택 사항: 현재 선택한 한 pair를 독립 HTML로 저장

위젯이 표시되지 않거나 세션이 끊긴 뒤에도 보고 싶을 때 사용합니다. Plotly 자체를 포함하므로 내려받은 HTML은 인터넷 없이 실행됩니다. 이 작은 대체 뷰어는 **지정한 한 layer/head만 포함**하며, 전체 layer/head 선택은 위 라이브 위젯에서 제공합니다. 모든 시간/128×128 원소와 signed/magnitude, slider/play, 고정 z-range, 회전/줌을 유지합니다. 기본 256토큰 한 pair는 원본 16 MiB + base64/라이브러리 때문에 HTML 약 26 MiB 이상이 됩니다.

아래 셀은 기본적으로 실행을 건너뜁니다. 필요하면 `EXPORT_HTML=True`로 바꾸십시오.

In [ ]:
EXPORT_HTML = False
HTML_LAYER, HTML_HEAD = LAYERS[0], HEADS[0]
if EXPORT_HTML:
    html_path = viewer.export_pair_html(
        trajectory, OUTPUT_DIR / f"state_explorer_L{HTML_LAYER}_H{HTML_HEAD}.html",
        layer_id=HTML_LAYER, head_id=HTML_HEAD,
    )
    print("HTML:", html_path)
    display(FileLink(str(html_path)))
    try:
        from google.colab import files
        files.download(str(html_path))
    except ImportError:
        pass


## 9. 기존 실험 1/3과 시간별 요약 통계 유지

모든 토큰을 수집했지만 SVD·양자화 진단은 기본적으로 기존 checkpoint(64,128,192,256 및 실제 마지막 토큰)에서만 계산합니다. 기본 1536개 행렬 전부에 CPU SVD를 실행하는 비용을 피하기 위한 **분석 설정**이며, NPZ와 3D 뷰어는 모든 토큰을 그대로 포함합니다.

원하면 아래 `CHECKPOINT_TOKENS`를 `range(1, len(trajectory["states"])+1)`로 바꾸어 모든 시점에 대해 분석할 수 있습니다. 기존 `state_snapshots.npz` 이름/키 형식도 내보내므로 이전 분석과 호환됩니다.

In [ ]:
states = trajectory_tools.checkpoint_snapshots(trajectory, CHECKPOINT_TOKENS)
np.savez_compressed(OUTPUT_DIR / "state_snapshots.npz", **{
    f"t{t:05d}_l{layer:02d}_h{head:02d}": state
    for (t, layer, head), state in states.items()
})
np.savez_compressed(OUTPUT_DIR / "token_ids.npz", input_ids=trajectory["token_ids"][None])
metrics, quantization = probe.analyze_snapshots(states, OUTPUT_DIR, bits=(8, 4))
metrics_df, quant_df = pd.DataFrame(metrics), pd.DataFrame(quantization)
display(metrics_df.round(5))


### 실험 1 — 분포와 spectrum 지표

다음 지표를 **center하지 않은 원래 state**에서 측정합니다. 따라서 SVD energy는 covariance의 분산 설명률과 구분합니다.

| 지표 | 의미 |
|---|---|
| E1, E4, E8 | 상위 1/4/8개 singular value 제곱합 / 전체 제곱합 |
| r90 | 전체 에너지의 90%에 도달하는 최소 rank |
| stable_rank | Frobenius norm 제곱 / 최대 singular value 제곱 |
| max_over_rms | 최대 절댓값 / 전체 원소의 RMS |
| top1pct_energy | 절댓값이 큰 상위 1% 원소가 차지하는 제곱합 비율 |
| u1_ipr, v1_ipr | 최대 singular vector의 원소 4제곱합; 특정 좌표에 집중될수록 큼 |
| leading_gap | (σ1 − σ2) / σ1; 최대 singular vector의 분리 정도 |

IPR은 단위벡터가 d개 좌표에 균일하게 퍼지면 1/d, 한 좌표에만 있으면 1입니다. **σ1과 σ2가 매우 가깝다면 u1/v1은 기저 선택과 수치 변동에 민감**하므로 IPR 한 값으로 해석하지 않습니다. 그 경우 row/column energy profile 또는 상위 singular subspace를 함께 봅니다.

### 실험 3 — 양자화 축 비교

원래 FP32 state를 모델 바깥에서 정수 코드로 반올림한 뒤 다시 실수로 복원하는 **static fake quantization**입니다. 양자화된 state를 다음 state update에 넣지 않습니다. 따라서 오차 누적 실험이나 모델 정확도 실험은 아닙니다.

| grouping | scale/min 공유 범위 |
|---|---|
| tensor | 하나의 head state 전체 |
| row | Key channel 하나의 행; Value 축으로 범위 계산 |
| column | Value channel 하나의 열; Key 축으로 범위 계산 |

모두 같은 affine uniform min-max 양자화를 사용합니다. **같은 nominal bits를 비교하지만 같은 총 메모리를 비교하는 것은 아닙니다.** row/column은 scale metadata가 더 많습니다. CSV에 FP32 min+scale 저장을 가정한 이론적 bits/entry도 기록합니다. 실제 비트 packing이나 가속 kernel 구현은 없으므로 메모리·속도 개선 수치로 사용하지 않습니다.

In [ ]:
display(quant_df[quant_df.tokens == quant_df.tokens.max()].round(6))
# Layer별로 그림을 나누어 Colab에서 글씨/그림이 지나치게 작아지지 않게 표시합니다.
for layer in LAYERS:
    subset = {key:value for key,value in states.items() if key[1] == layer}
    paths = probe.make_figures(
        subset,
        [r for r in metrics if r["layer"] == layer],
        [r for r in quantization if r["layer"] == layer],
        OUTPUT_DIR / f"layer_{layer}", plot_head=HEADS[0],
    )
    for path in paths:
        display(Image(filename=path, width=1100))


### 연구 제안서에 넣을 때의 해석

1. 먼저 여러 문서에서 같은 layer/head의 결과를 비교합니다. 이 노트북의 default 예시 한 문서는 예비 관찰입니다.
2. 상위 singular energy가 높아도 element-wise outlier가 자동으로 생기지는 않습니다. magnitude, row/column RMS, U/V localization을 함께 봅니다.
3. channel별 양자화 오차가 줄어들면 축별 범위 차이가 중요한 단서입니다. 그 사실만으로 outlier 형성 원인이 밝혀진 것은 아닙니다.
4. 다음 단계는 state update의 실제 K/V 및 gate를 수집하여 원래 recurrence를 replay하는 것입니다. **Projection 직후가 아닌 conv+SiLU 이후 입력**을 사용하고, kernel 내부 Q/K L2 normalization과 query scaling을 재현해야 합니다.
5. 이후에는 실제 query로 core output 오차를 측정하고, 양자화된 state를 반복 갱신하는 오차 누적 실험을 분리해서 수행합니다.

**그림 캡션 예시:** “Pretrained Qwen3.5-0.8B의 선택된 Gated DeltaNet layer에서 관찰한 recurrent state의 크기 분포와 singular spectrum. 단일 입력 문서에 대한 예비 관찰이며, outlier의 발생 원인에 대한 인과적 결론은 제시하지 않는다.” 실제 실행 후 모델 revision, 문서 수, token 수, layer/head 번호를 함께 적습니다.

그림 1의 heatmap은 layer마다 독립적인 색 범위를 사용합니다. 서로 다른 layer의 색 자체를 절대 크기 비교에 사용하지 않습니다.

## 10. GPU 메모리·속도와 문제 해결

**CPU trajectory 저장량 = T × layer 수 × head 수 × 128 × 128 × 4 bytes.** 한 pair/256토큰은 16 MiB, 기본 6개 pair는 96 MiB, 6개 pair/1024토큰은 384 MiB입니다. NPZ 왕복 검증 때 state 배열을 다시 읽으므로 순간적으로 저장량의 **약 2–3배 CPU RAM**과 Python/모델/브라우저 메모리 여유가 필요합니다. `CPU_STATE_BUDGET_MIB`는 trajectory 배열 자체에 대한 상한입니다. 디스크 압축률은 값에 따라 달라집니다.

GPU에는 전체 모델과 현재 cache가 필요합니다. 이 모델의 18개 linear layer × 16 heads FP32 recurrent cache는 합계 약 **18 MiB**이며, 저장할 layer/head를 줄여도 이 모델 내부 cache는 줄지 않습니다. full-attention KV는 토큰 수에 비례하여 증가하고 convolution cache 및 임시 activation도 필요합니다. 선택한 과거 state만 CPU에 쌓아 GPU에 T개 이력을 보관하지 않습니다. 실제 allocated와 reserved 최고값은 위 수집 결과에 기록되며, 고정된 VRAM 사용량을 보장하지 않습니다.

매 토큰 1회 forward + 선택 state의 동기 CPU 복사가 필요하므로 원본의 64토큰 chunk 처리보다 느립니다. 기록한 tokens/s는 관찰 overhead를 포함하며 일반 추론 throughput이 아닙니다. layer/head 수를 줄이면 복사/저장 비용은 줄지만 모든 모델 layer의 계산은 계속합니다. 기본 256토큰부터 시작하고 실제 측정 시간을 확인한 뒤 확장하십시오.

| 증상 | 조치 |
|---|---|
| GPU OOM | 새 GPU 런타임에서 다른 모델/이전 변수를 정리하고 토큰 수를 줄입니다. 수집은 이미 한 토큰 forward이며 별도의 CHUNK_SIZE 설정이 없습니다. |
| CPU RAM/NPZ 저장 부족 | `MAX_TOKENS`, `LAYERS`, `HEADS`를 줄입니다. 기존 trajectory와 explorer를 여러 개 유지하지 않습니다. |
| BF16/T4 속도 문제 | 이 로더는 native 지원 여부를 검사해 T4에서는 FP16을 선택합니다. 과거 BF16 결과와는 dtype 차이를 기록합니다. |
| 버전/API 오류 | 설치 셀의 고정 버전을 확인하고 runtime을 재시작합니다. 오류를 무시하고 `.float()`만 덧붙이지 않습니다. |
| `fla`/`causal_conv1d` 감지 | 이 커널을 설치하지 않은 새 Colab runtime에서 실행합니다. native reference와 커널 결과를 몰래 혼합하지 않습니다. |
| 검증 `needs_review` | `validation_comparisons.csv`에서 방법/시점/layer별 절대·상대 오차를 확인합니다. kernel, dtype, 버전, 실행 설정을 점검합니다. 결과를 검증 완료로 표시하지 않습니다. |
| 위젯 공백 | custom widget manager 셀을 다시 실행하고 출력/브라우저 새로고침. 설치 전 import했다면 재시작. 대안은 한 pair HTML export입니다. |
| 재생 지연 | Playback을 1.5 s로, Grid를 64×64로 변경. 데이터는 128×128 그대로 남습니다. |

추가 문서는 `TEXT`와 `OUTPUT_DIR`을 바꾸고 수집부터 다시 실행합니다. **LAYERS/HEADS 등 실험 설정을 바꿀 때도 새 OUTPUT_DIR을 사용**하십시오. 같은 폴더를 재사용하면 이전 layer 그림/선택 HTML이 ZIP에 남을 수 있습니다. 각 수집 함수 호출은 빈 cache에서 시작합니다. 파일을 내려받을 때 입력 문서와 token IDs도 포함된다는 점을 기억하십시오.

## 11. 결과 다운로드

NPZ 하나만 전달하면 모든 state와 token/layer/head·실행 metadata를 재사용할 수 있습니다. 검증과 기존 실험 결과를 함께 전달하려면 ZIP을 받으십시오. 버튼을 눌렀을 때 다운로드하므로 `모두 실행`이 여러 자동 다운로드를 발생시키지 않습니다.

In [ ]:
import shutil
import ipywidgets as widgets
archive = Path(shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR))

def download_result(path):
    try:
        from google.colab import files
        files.download(str(path))
    except ImportError:
        display(FileLink(str(path)))

npz_button = widgets.Button(description="Download NPZ", icon="download")
zip_button = widgets.Button(description="Download all (ZIP)", icon="download")
npz_button.on_click(lambda _: download_result(npz_path))
zip_button.on_click(lambda _: download_result(archive))
display(widgets.HBox([npz_button, zip_button]))
print("NPZ:", npz_path.resolve())
print("ZIP:", archive.resolve())


## 12. 나중에 NPZ만 다시 보기 (선택 사항, GPU 불필요)

새 세션에서 1–2절(설치/공통 함수 쓰기)까지만 실행한 후 아래 셀을 사용하면 모델을 로드하지 않습니다. Colab 왼쪽 파일 패널에서 NPZ를 먼저 업로드하고 `REOPEN_PATH`에 그 경로를 넣으십시오. 이미 수집한 trajectory와 동시에 로드하면 RAM이 늘어납니다. 기본은 건너뜁니다.

In [ ]:
REOPEN_PATH = ""  # 예: "/content/recurrent_state_trajectory.npz"
if REOPEN_PATH:
    import state_trajectory as trajectory_tools
    import state_viewer as viewer
    from IPython.display import display
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
    if "explorer" in globals():
        viewer.close_explorer(explorer)
    trajectory = trajectory_tools.load_trajectory(REOPEN_PATH)
    explorer = viewer.build_explorer(trajectory)
    display(explorer)


## 공식 구현 근거와 변경 이력

- [Qwen3.5-0.8B 모델 카드](https://huggingface.co/Qwen/Qwen3.5-0.8B)
- [고정 revision의 config](https://huggingface.co/Qwen/Qwen3.5-0.8B/blob/2fc06364715b967f1860aea9cf38778875588b17/config.json)
- [Transformers v5.24.0 Qwen3.5 forward/FP32 recurrence](https://github.com/huggingface/transformers/blob/v5.24.0/src/transformers/models/qwen3_5/modeling_qwen3_5.py)
- [동일 버전 cache API와 in-place update](https://github.com/huggingface/transformers/blob/v5.24.0/src/transformers/cache_utils.py)
- [Hub/로컬 커널 fallback 처리](https://github.com/huggingface/transformers/blob/v5.24.0/src/transformers/integrations/hub_kernels.py)
- [Plotly FigureWidget](https://plotly.com/python/figurewidget/), [3D surface](https://plotly.com/python/3d-surface-plots/), [3D scene/camera](https://plotly.com/python/reference/layout/scene/)
- [ipywidgets Play/slider](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html)

변경일: **2026-11-24 (Asia/Seoul)**. `/mnt/data`는 현재 데스크톱 환경에 없어서 다운로드 폴더의 `qwen_recurrent_state_gpu_bundle/qwen_recurrent_state_gpu.ipynb`를 원본으로 읽었습니다. 이전 실험 결과를 새 수집 결과로 복제하지 않았습니다. 원본 notebook SHA-256: `e170767f4b5984330657bdaf37b263e5d17643b7b0380c8db086aa17d4958392`.

**개발 검증:** 작은 무작위 초기화 CPU Qwen3.5TextModel(실제 Transformers 5.24.0, 128×128 state)에서 native cache와 새 수집 함수 통합 테스트 통과. 잘못된 layer/head, FP16 state, 복사 별칭, cache 초기화 및 저장 데이터 변조도 검사했습니다. NPZ exact roundtrip 및 viewer 컨트롤 검사를 통과했습니다. 이 테스트는 pretrained 0.8B GPU/Colab 검증을 대체하지 않습니다.